# Customer segmentation → personas (K-Means)

Groups costmers by **who they are** (profile) and **how they buy and engage**
(behavior) to build business personas, and reads offer response
within each one. Modeling is at **costmer level** (one row per
`account_id`) and **descriptive**: uses the full test window, so it names
segments and explains results — never becomes an X-learner feature (that would be
leakage, G2).

All logic lives in `src/clustering/`; here we only import, run, and read.
Spark loads and reduces to costmer grain; pandas does the analysis.

Sections: 1. Setup & load · 2. Customer grain · 3. Variable treatment ·
4. Choosing k · 5. Fit · 6. Cluster profiles · 7. Personas ·
8. Response by persona · 9. Synthesis.

## 1. Setup & load

Spark loads raw events and the processed grain; the executive theme from `src/viz.py` is applied once.

In [ ]:
import os, sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "config.yaml").exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from src import clustering, viz
from src.config import load
from src.io import parse_events
from src.pipeline import build_spark

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)
viz.setup_theme()

cfg = load()
spark = build_spark(cfg, app_name="clustering")
spark.sparkContext.setLogLevel("ERROR")

events = parse_events(spark, cfg).cache()
processed = spark.read.parquet(str(cfg.processed_dir)).cache()
print(f"events: {events.count():,} | processed rows: {processed.count():,}")

## 2. Customer grain

One row per `account_id`: profile, purchases (full window), engagement, and observed response. Only `cfg.cluster_features` enter the fit; `conv_rate`/`margem` are for reading.

In [ ]:
clientes = clustering.build_client_frame(processed, events)
print(f"{len(clientes):,} clientes | {int(clientes['identity_missing'].sum()):,} with missing identity")
print(f"fitting features: {cfg.cluster_features}")
clientes.head()

In [ ]:
# Univariate profile of fitting features, in original units — the tails
# monetary segment and sentinel segment (no age/limit) that motivate treatment.
resumo = clientes[list(cfg.cluster_features)].describe(percentiles=[.25, .5, .75, .99]).T
resumo["nulos"] = clientes[list(cfg.cluster_features)].isna().sum().values
resumo.round(2)

## 3. Variable treatment

Three decisions make Euclidean distance meaningful (all in `clustering.design_matrix`): **no imputation** of the `identity_missing` segment (it is already a segment, excluded from the fit), **`log1p`** on monetary tails (`spend_total`/`txn_count`/`avg_ticket`), and **z-score** afterward (features in different units).

In [ ]:
dm = clustering.design_matrix(clientes, cfg)
print(f"fitting matrix: {dm.X.shape[0]:,} costmers × {dm.X.shape[1]} features")
print(f"log1p applied to: {dm.log_columns}")
print(f"standardization — max mean |{np.abs(dm.X.mean(axis=0)).max():.2e}|, "
      f"max std dev |{np.abs(dm.X.std(axis=0) - 1).max():.2e}| from 1")

In [ ]:
# spend_total before/after log1p: the tail that would dominate Euclidean distance.
completos = clientes.loc[clientes["identity_missing"] == 0]
h_bruto = np.histogram(completos["spend_total"], bins=cfg.histogram_bins)
h_log = np.histogram(np.log1p(completos["spend_total"]), bins=cfg.histogram_bins)
paineis = [
    {"title": "spend_total (raw): long tail",
     "x": ((h_bruto[1][:-1] + h_bruto[1][1:]) / 2).tolist(), "y": h_bruto[0].tolist()},
    {"title": "log1p(spend_total): symmetric",
     "x": ((h_log[1][:-1] + h_log[1][1:]) / 2).tolist(), "y": h_log[0].tolist()},
]
viz.faceted(
    paineis, kind="bar", cols=2, row_height=340,
    title="log1p removes the tail that would dictate centroids",
    subtitle=f"n={len(completos):,} costmers with complete profile",
).show()

## 4. Choosing k

Scan of `k` with four indices on the same standardized matrix: inertia (elbow), silhouette (decision criterion, ↑), Davies-Bouldin (↓), and Calinski-Harabasz (↑). The criterion stays visible — never just the winner.

In [ ]:
scan = clustering.scan_k(dm.X, cfg)
k = clustering.choose_k(scan)
print(f"k chosen (max silhouette): {k}")
scan.round(4)

In [ ]:
# Four indices side by side; line marks chosen k. Quantities without
# common unit never go on dual y-axis — each panel has its own scale.
melhor_sil = scan.loc[scan["silhouette"].idxmax(), "silhouette"]
faixa_sil = scan["silhouette"].max() - scan["silhouette"].min()
ks = scan["k"].tolist()
viz.faceted(
    [
        {"title": "inertia (elbow, ↓)", "x": ks, "y": scan["inercia"].tolist()},
        {"title": "silhouette (criterion, ↑)", "x": ks, "y": scan["silhouette"].tolist()},
        {"title": "Davies-Bouldin (↓)", "x": ks, "y": scan["davies_bouldin"].tolist()},
        {"title": "Calinski-Harabasz (↑)", "x": ks, "y": scan["calinski_harabasz"].tolist()},
    ],
    kind="line", cols=2, vline=k, xlabel="k (number of clusters)", row_height=320,
    title=f"k = {k} maximizes silhouette ({melhor_sil:.3f}); the curve is shallow (range {faixa_sil:.3f})",
    subtitle="segments are cuts along a value gradient, not distinct species",
).show()

## 5. Fit

K-Means with the chosen k, deterministic via `cfg.seed`. The missing-identity segment, excluded from the fit, is reattached as its own segment (not imputed).

In [ ]:
rotulos = clustering.fit(dm.X, dm.account_ids, k, cfg)
segmentos = clustering.assign_segments(clientes, rotulos)
segmentos.groupby("segmento").size().rename("clientes").reset_index()

## 6. Cluster profiles

Mean of each feature per segment, in original units. The heatmap shows z-score across segments (raw means are not comparable across reals/years/days); raw value is annotated in the cell.

In [ ]:
perfil = clustering.profile_segments(segmentos)
perfil.round(2)

In [ ]:
# z-score per column across segments; raw value overlaid. view_rate/conv_rate
# were excluded from fit (they are the response), shown here for reading only.
colunas_perfil = [c for c in list(cfg.cluster_features) + ["conv_rate"] if c in perfil.columns]
valores = perfil.set_index("segmento")[colunas_perfil].astype("float64")
z = (valores - valores.mean()) / valores.std(ddof=0).replace(0, np.nan)
viz.heatmap(
    z, diverging=True, annotate=True, text=valores.to_numpy(), text_template="%{text:.4g}",
    colorbar_title="z", reverse_y=False,
    title="Each segment is a point on the value and engagement gradient",
    subtitle="z-score across segments · raw value annotated",
).show()

## 7. Personas

K-Means labels (`cluster 0/1/…`) are arbitrary; `name_personas` maps them to business labels derived from the profile itself (ordered by economic value), plus the segment without registration.

In [ ]:
personas = clustering.name_personas(perfil)
cartao = personas[["persona", "segmento", "clientes", "fracao"] +
                  [c for c in ["spend_total", "avg_ticket", "txn_count", "tenure_days",
                               "credit_card_limit", "age", "view_rate", "conv_rate"]
                   if c in personas.columns]]
cartao.sort_values("clientes", ascending=False).round(2).reset_index(drop=True)

## 8. Response by persona

How each persona responds to each offer type — the business reading that closes segmentation. Observational (mean of what happened), not the causal effect of sending; explicit denominators.

In [ ]:
nome_por_segmento = personas.set_index("segmento")["persona"].to_dict()
resposta = clustering.segment_response(
    processed, spark.createDataFrame(segmentos[["account_id", "segmento"]]))
resposta["persona"] = resposta["segmento"].map(nome_por_segmento)
resposta.round(4)

In [ ]:
# Margin per send (revenue − cost) / sends, one bar per offer type,
# grouped by persona — where a uniform policy pays off.
melhor = resposta.loc[resposta["margem_por_envio"].idxmax()]
pior = resposta.loc[resposta["margem_por_envio"].idxmin()]
tipos = sorted(resposta["offer_type"].unique())
margem_wide = (
    resposta.pivot(index="persona", columns="offer_type", values="margem_por_envio")
    .reset_index()
)
viz.grouped_bars(
    margem_wide, category="persona", series=tipos, value_label="%{y:.1f}",
    ylabel="margin per send (R$)",
    title=f"Margin per send ranges from {pior['margem_por_envio']:.1f} ({pior['persona']}, {pior['offer_type']}) "
          f"to {melhor['margem_por_envio']:.1f} ({melhor['persona']}, {melhor['offer_type']})",
    subtitle="observational mean per send · not the causal effect of sending",
).show()

## 9. Synthesis

Personas and what each one says for offer decisions. Descriptive — the causal effect of sending is what spec 02 estimates; here we fix who is who and where observed response varies.

In [ ]:
sintese = personas[["persona", "clientes", "fracao"]].copy()
sintese = sintese.merge(
    resposta.groupby("persona").agg(
        margem_media=("margem_por_envio", "mean"),
        melhor_oferta=("offer_type", lambda s: resposta.loc[s.index].sort_values(
            "margem_por_envio").iloc[-1]["offer_type"]),
    ).reset_index(),
    on="persona", how="left",
)
sintese.sort_values("clientes", ascending=False).round(3).reset_index(drop=True)